In [1]:
# =========================
# RAG EMBEDDINGS + RETRIEVAL (ONE CELL)
# =========================

import torch
import numpy as np
import json
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# LOAD WIKI DATA
# -------------------------
with open("wiki_kb.json") as f:
    kb = json.load(f)

# build contexts (use your structure)
contexts = [
    f"Species: {item['species']}\n{item['text']}"
    for item in kb
]

print(f"Loaded {len(contexts)} contexts")

# -------------------------
# LOAD EMBEDDING MODEL
# -------------------------
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
embedding_model = AutoModel.from_pretrained(embedding_model_name).to(device)

# -------------------------
# EMBEDDING FUNCTION
# -------------------------
def compute_embeddings(texts, model, tokenizer):
    embeddings = []

    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # mean pooling
        embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        embeddings.append(embedding)

    return np.vstack(embeddings)

# -------------------------
# BUILD EMBEDDINGS
# -------------------------
print("Computing embeddings...")
context_embeddings = compute_embeddings(contexts, embedding_model, embedding_tokenizer)

print("Embeddings shape:", context_embeddings.shape)

# -------------------------
# RETRIEVAL FUNCTION
# -------------------------
def retrieve_top_context(query, context_embeddings, contexts, model, tokenizer):
    query_embedding = compute_embeddings([query], model, tokenizer)

    dot_products = np.dot(context_embeddings, query_embedding.T).squeeze()
    context_norms = np.linalg.norm(context_embeddings, axis=1)
    query_norm = np.linalg.norm(query_embedding)

    similarities = dot_products / (context_norms * query_norm)

    top_idx = np.argmax(similarities)
    return contexts[top_idx]

# -------------------------
# TEST
# -------------------------
query = "What environment do Korean live in?"

result = retrieve_top_context(
    query,
    context_embeddings,
    contexts,
    embedding_model,
    embedding_tokenizer
)

print("\n--- RESULT ---\n")
print(result)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Loaded 103 contexts


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Computing embeddings...
Embeddings shape: (103, 384)

--- RESULT ---

Species: abies koreana
Abies koreana: Abies koreana, the Korean fir, is a species of fir native to the higher mountains of South Korea, including Jeju Island. It grows at altitudes of 1,000–1,900 metres (3,300–6,200 ft) in temperate rainforest with high rainfall and cool, humid summers, and heavy winter snowfall.


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-2")
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-2",
    torch_dtype=torch.float32,
    trust_remote_code=True
).to(device)

model.config.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [9]:
# RAG-style answer (clean response)
def answer_query(query):
    context = retrieve_top_context(
        query, context_embeddings, contexts, embedding_model, embedding_tokenizer
    )
    prompt = f"""Answer the question using the information below.
Context:
{context}
Question: {query}
Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_length = inputs["input_ids"].shape[1]  # track where prompt ends

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode ONLY the newly generated tokens, not the prompt
    new_tokens = outputs[0][prompt_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Stop at newline to avoid "Exercise 2..." continuations
    answer = answer.split("\n")[0].strip()
    return answer


# test
queries = [
    "What is Korean fir?",
    "Tell me about cornus florida",
    "Where is abies koreana found?",
    "What kind of plant is magnolia stellata?",
    "Tell me about Korean fir",
    "Where is Korean fir found?",
    "What kind of plant is Korean fir?",
    "What is the name of Korean fir"
]
for q in queries:
    print(f"\n=== QUERY: {q} ===")
    print(answer_query(q))


=== QUERY: What is Korean fir? ===
Korean fir is a species of fir native to the higher mountains of South Korea, including Jeju Island.

=== QUERY: Tell me about cornus florida ===
Cornus florida, also known as the flowering dogwood or American dogwood, is a species of flowering tree in the family Cornaceae native to eastern North America and northern Mexico. It is commonly found in residential and public areas due to its showy bracts and unique bark structure.

=== QUERY: Where is abies koreana found? ===
Abies koreana is found in the higher mountains of South Korea, including Jeju Island.

=== QUERY: What kind of plant is magnolia stellata? ===
Magnolia stellata is a slow-growing deciduous shrub or small tree native to Japan.

=== QUERY: Tell me about Korean fir ===
Korean fir is a species of fir native to the higher mountains of South Korea, including Jeju Island. It grows at altitudes of 1,000–1,900 metres (3,300–6,200 ft) in temperate rainforest with high rainfall and cool, humid